# 🔓 Solutions: Unsupervised Learning

> **Try first, peek second.** You learn far more by struggling with a problem for ten minutes than by reading its answer for ten seconds. Use these solutions to *compare* your approach, not to replace it. There is often more than one correct way.


In [1]:
import sys
sys.path.append("..")
from course_utils import *
setup()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.datasets import make_moons, load_digits

### 🧩 Problem 3.1: Segment the customers &nbsp; ⭐

1. Load `customers.csv` and use the four features `["age", "annual_income_k", "spending_score", "visits_per_month"]` (ignore `segment_truth`).
2. **Standardise** them with `StandardScaler`.
3. Fit `KMeans(n_clusters=4, n_init=10, random_state=0)`.
4. Store the cluster label of every customer in `labels`.

In [2]:
customers = load("customers.csv")
features = ["age", "annual_income_k", "spending_score", "visits_per_month"]

Xs = StandardScaler().fit_transform(customers[features])
labels = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(Xs)
print(np.bincount(labels))

[100 101  99 100]


In [3]:
# ✔ Self-check
truth = load("customers.csv")["segment_truth"]
check("one label per customer", lambda: len(labels) == 400)
check("exactly 4 clusters", lambda: len(set(labels)) == 4)
check("clusters match the hidden segments well (ARI > 0.9)", lambda: adjusted_rand_score(truth, labels) > 0.9)

✅ PASSED   one label per customer
✅ PASSED   exactly 4 clusters
✅ PASSED   clusters match the hidden segments well (ARI > 0.9)


True

**🧑‍🏫 Walkthrough.** Scale → cluster → read `labels_`. The `n_init=10` setting runs K-Means from 10 different random starts and keeps the best, protecting you from an unlucky initialisation (K-Means only finds a *local* optimum). `random_state=0` makes results repeatable.

### 🧩 Problem 3.2: Choose k with the silhouette score &nbsp; ⭐⭐

For every `k` from **2 to 8**, fit K-Means on the **standardised** customer features (use `n_init=10, random_state=0`) and compute the silhouette score.

Store the scores in a dictionary `sil_scores` (`{k: score}`) and the *k* with the highest score in `best_k`.

In [4]:
customers = load("customers.csv")
Xs = StandardScaler().fit_transform(customers[["age", "annual_income_k", "spending_score", "visits_per_month"]])

sil_scores = {}
for k in range(2, 9):
    m = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xs)
    sil_scores[k] = silhouette_score(Xs, m.labels_)

best_k = max(sil_scores, key=sil_scores.get)
print({k: round(v, 3) for k, v in sil_scores.items()}, "→ best k =", best_k)

{2: 0.495, 3: 0.522, 4: 0.535, 5: 0.474, 6: 0.431, 7: 0.388, 8: 0.319} → best k = 4


In [5]:
# ✔ Self-check
check("scores for k = 2..8", lambda: sorted(sil_scores) == list(range(2, 9)))
check("every score is between -1 and 1", lambda: all(-1 <= v <= 1 for v in sil_scores.values()))
check("best_k is the k with the max score", lambda: best_k == max(sil_scores, key=sil_scores.get))
check("best_k = 4 (the real number of segments)", lambda: best_k == 4)

✅ PASSED   scores for k = 2..8
✅ PASSED   every score is between -1 and 1
✅ PASSED   best_k is the k with the max score
✅ PASSED   best_k = 4 (the real number of segments)


True

**🧑‍🏫 Walkthrough.** The silhouette score peaks at the true number of segments. Notice we *never used the truth* to find k: only the geometry of the data. In practice silhouette scores may be nearly flat (e.g. 0.31 vs 0.33); then business context should break the tie.

### 🧩 Problem 3.3: Name the segments (cluster profiling) &nbsp; ⭐⭐

Using your 4-cluster K-Means on the standardised customers:

1. Create a DataFrame `profile` with the **mean `annual_income_k` and mean `spending_score` of each cluster** (index = cluster id).
2. Store in `vip_cluster` the **id** of the cluster with the highest *average spending score among clusters whose average income is above 70*.

In [6]:
customers = load("customers.csv")
Xs = StandardScaler().fit_transform(customers[["age", "annual_income_k", "spending_score", "visits_per_month"]])
customers["cluster"] = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(Xs)

profile = customers.groupby("cluster")[["annual_income_k", "spending_score"]].mean()
rich = profile[profile["annual_income_k"] > 70]
vip_cluster = rich["spending_score"].idxmax()
print(profile.round(1)); print("VIP cluster id:", vip_cluster)

         annual_income_k  spending_score
cluster                                 
0                   86.6            22.3
1                   31.1            75.0
2                   89.6            80.7
3                   32.7            24.7
VIP cluster id: 2


In [7]:
# ✔ Self-check
check("profile has 4 rows and the 2 requested columns", lambda: profile.shape == (4, 2))
check("vip cluster has high income (> 70)", lambda: profile.loc[vip_cluster, "annual_income_k"] > 70)
check("vip cluster has high spending (> 65)", lambda: profile.loc[vip_cluster, "spending_score"] > 65)

✅ PASSED   profile has 4 rows and the 2 requested columns
✅ PASSED   vip cluster has high income (> 70)
✅ PASSED   vip cluster has high spending (> 65)


True

**🧑‍🏫 Walkthrough.** `groupby(...).mean()` is the workhorse of cluster interpretation. Two clusters have high income: one spends little (the "savers"), one spends a lot (the VIPs). Cluster ids are arbitrary labels, which is why we find the VIP cluster by its *properties* rather than hard-coding "cluster 1".

### 🧩 Problem 3.4: How many components? (PCA on wine) &nbsp; ⭐⭐

Load `wine.csv` (178 wines, 13 chemical measurements; the `target` column is the grape variety; *do not use it as a feature*).

1. Standardise the 13 features.
2. Fit a full `PCA()`.
3. Store `var_first_two`, the fraction of variance explained by the **first two** components, and `n_90`, the **smallest number of components** whose cumulative explained variance is **at least 90%**.

In [8]:
wine = load("wine.csv")
X_wine = wine.drop(columns="target")

Xw = StandardScaler().fit_transform(X_wine)
pca = PCA().fit(Xw)
ratios = pca.explained_variance_ratio_
var_first_two = ratios[:2].sum()
n_90 = int(np.argmax(np.cumsum(ratios) >= 0.90)) + 1
print(f"first two PCs: {var_first_two:.1%};  components for 90%: {n_90} of {X_wine.shape[1]}")

first two PCs: 55.4%;  components for 90%: 8 of 13


In [9]:
# ✔ Self-check
_r = PCA().fit(StandardScaler().fit_transform(load("wine.csv").drop(columns="target"))).explained_variance_ratio_
check("variance of first two PCs is about 55%", lambda: abs(var_first_two - _r[:2].sum()) < 1e-6)
check("n_90 is correct", lambda: n_90 == int(np.argmax(np.cumsum(_r) >= 0.90)) + 1)
check("n_90 is smaller than the 13 original features", lambda: n_90 < 13)

✅ PASSED   variance of first two PCs is about 55%
✅ PASSED   n_90 is correct
✅ PASSED   n_90 is smaller than the 13 original features


True

**🧑‍🏫 Walkthrough.** The first two components explain only about 55% of the variance: wine chemistry is genuinely multi-dimensional, and a 2-D plot hides a lot. You need **8 of 13** components to retain 90%. *Always standardise before PCA*; otherwise the feature with the biggest numbers (proline, in the hundreds) would dominate PC1 purely because of its units.

### 🧩 Problem 3.5: Market-basket analysis &nbsp; ⭐⭐

Ten shoppers' baskets are given below. Compute for the rule **milk → bread**:

* `sup_both`: the support of {milk, bread}
* `conf`: the confidence of milk → bread
* `lift`: the lift of milk → bread

Then decide: does buying milk really make bread *more* likely? Store `True`/`False` in `milk_boosts_bread`.

In [10]:
baskets = [{"milk", "bread", "eggs"}, {"milk", "bread"}, {"bread", "butter"}, {"milk", "eggs"},
           {"milk", "bread", "butter"}, {"bread", "eggs"}, {"milk", "bread", "eggs", "butter"},
           {"eggs"}, {"milk", "bread"}, {"bread", "butter"}]

def support(items):
    return sum(set(items) <= b for b in baskets) / len(baskets)

sup_both = support({"milk", "bread"})
conf = sup_both / support({"milk"})
lift = conf / support({"bread"})
milk_boosts_bread = lift > 1.1
print(f"support={sup_both:.2f}  confidence={conf:.3f}  lift={lift:.3f}  boosts? {milk_boosts_bread}")

support=0.50  confidence=0.833  lift=1.042  boosts? False


In [11]:
# ✔ Self-check
check("support of {milk, bread} = 0.5", lambda: abs(sup_both - 0.5) < 1e-9)
check("confidence = 5/6", lambda: abs(conf - 5 / 6) < 1e-9)
check("lift = (5/6)/0.8", lambda: abs(lift - (5 / 6) / 0.8) < 1e-9)
check("verdict: only a tiny boost (lift < 1.1) → False", lambda: milk_boosts_bread is False)

✅ PASSED   support of {milk, bread} = 0.5
✅ PASSED   confidence = 5/6
✅ PASSED   lift = (5/6)/0.8
✅ PASSED   verdict: only a tiny boost (lift < 1.1) → False


True

**🧑‍🏫 Walkthrough.** Confidence is a high-sounding **83%**: but bread is in 80% of *all* baskets, so almost everyone buys bread anyway. **Lift ≈ 1.04** exposes it: milk barely changes the odds. This is why lift, not confidence, is the honest measure. (*Is 1.1 a magic threshold? No, it's a judgement call; the point is that 1.04 is nearly 1.*)

### 🧩 Problem 3.6: K-Means from scratch &nbsp; ⭐⭐⭐

Implement `my_kmeans(X, k, n_iter=100, n_init=10, seed=0)` using **only NumPy**. It should return `(labels, centres)`.

* Initialise centres by picking `k` random data points.
* Repeat *assign → update* until the centres stop moving (or `n_iter` rounds).
* Run `n_init` times from different random starts and keep the run with the lowest **inertia** (sum of squared distances to the assigned centre).

*(Edge case: if a cluster becomes empty, keep its old centre.)*

In [12]:
def my_kmeans(X, k, n_iter=100, n_init=10, seed=0):
    best = (np.inf, None, None)
    for init in range(n_init):
        rng = np.random.default_rng(seed + init)
        centres = X[rng.choice(len(X), size=k, replace=False)].copy()      # random starting centres
        for _ in range(n_iter):
            dists = ((X[:, None, :] - centres[None, :, :]) ** 2).sum(axis=2)   # (n_points, k)
            labels = dists.argmin(axis=1)                                       # assign step
            new = np.array([X[labels == j].mean(axis=0) if (labels == j).any() else centres[j]
                            for j in range(k)])                                 # update step
            if np.allclose(new, centres):
                break
            centres = new
        inertia = dists[np.arange(len(X)), labels].sum()
        if inertia < best[0]:
            best = (inertia, labels, centres)
    return best[1], best[2]

_c = load("customers.csv")
_X = StandardScaler().fit_transform(_c[["age", "annual_income_k", "spending_score", "visits_per_month"]])
lab, cen = my_kmeans(_X, 4)
print("cluster sizes:", np.bincount(lab))

cluster sizes: [ 99 100 101 100]


In [13]:
# ✔ Self-check
_c = load("customers.csv")
_X = StandardScaler().fit_transform(_c[["age", "annual_income_k", "spending_score", "visits_per_month"]])
_labels, _centres = my_kmeans(_X, 4)
check("returns 400 labels and 4 centres", lambda: len(_labels) == 400 and _centres.shape == (4, 4))
check("recovers the hidden segments (ARI > 0.9)", lambda: adjusted_rand_score(_c["segment_truth"], _labels) > 0.9)

✅ PASSED   returns 400 labels and 4 centres
✅ PASSED   recovers the hidden segments (ARI > 0.9)


True

**🧑‍🏫 Walkthrough.** That's the entire K-Means algorithm in ~12 lines. The multi-start loop matters: a single random start can place two centres inside one blob and leave another blob uncovered (a *local optimum*). Keeping the lowest-inertia run is exactly what `n_init` does in scikit-learn.